In [1]:
import copy
import math

# Định nghĩa các hằng số
X = "X"
O = "O"
EMPTY = None

def initial_state():
    """Khởi tạo ma trận 3x3 trống"""
    return [[EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY]]

def player(board, user_char, turn_count):
    """Xác định lượt đi: Người chơi luôn đi lượt chẵn (0, 2, 4...)"""
    ai_char = O if user_char == X else X
    return user_char if turn_count % 2 == 0 else ai_char

def actions(board):
    """Trả về tập hợp các tọa độ (hàng, cột) còn trống"""
    return {(i, j) for i in range(3) for j in range(3) if board[i][j] == EMPTY}

def result(board, action, p_char):
    """Thực hiện nước đi và trả về trạng thái mới của bàn cờ"""
    i, j = action
    new_board = copy.deepcopy(board)
    new_board[i][j] = p_char
    return new_board

def winner(board):
    """Kiểm tra điều kiện thắng trên hàng, cột và đường chéo"""
    for i in range(3):
        # Kiểm tra hàng ngang và hàng dọc
        if board[i][0] == board[i][1] == board[i][2] != EMPTY: return board[i][0]
        if board[0][i] == board[1][i] == board[2][i] != EMPTY: return board[0][i]
    # Kiểm tra hai đường chéo
    if board[0][0] == board[1][1] == board[2][2] != EMPTY: return board[0][0]
    if board[0][2] == board[1][1] == board[2][0] != EMPTY: return board[0][2]
    return None

def terminal(board):
    """Trạng thái kết thúc: Có người thắng hoặc bàn cờ đã đầy"""
    return winner(board) is not None or all(EMPTY not in row for row in board)

def utility(board, user_char):
    """Trả về giá trị điểm số cho trạng thái kết thúc"""
    res = winner(board)
    ai_char = O if user_char == X else X
    if res == ai_char: return 10
    if res == user_char: return -10
    return 0

# --- CÀI ĐẶT THUẬT TOÁN ALPHA-BETA PRUNING ---

def alpha_beta_search(board, user_char):
    """Hàm tìm kiếm nước đi tối ưu nhất cho AI"""
    if terminal(board):
        return None

    ai_char = O if user_char == X else X
    best_val = -math.inf
    best_move = None
    alpha = -math.inf
    beta = math.inf

    # AI đóng vai trò là Maximizer (Tìm giá trị Max)
    for action in actions(board):
        # Gọi xuống min_value vì lượt tiếp theo là của đối thủ
        val = min_value(result(board, action, ai_char), user_char, alpha, beta)
        if val > best_val:
            best_val = val
            best_move = action
        # Cập nhật Alpha (Giá trị tốt nhất Maximizer tìm được)
        alpha = max(alpha, best_val)
    return best_move

def max_value(state, user_char, alpha, beta):
    if terminal(state):
        return utility(state, user_char)

    v = -math.inf
    ai_char = O if user_char == X else X
    for action in actions(state):
        v = max(v, min_value(result(state, action, ai_char), user_char, alpha, beta))
        alpha = max(alpha, v)
        # Cắt tỉa: Nếu giá trị tìm được đã lớn hơn Beta của lớp trên, dừng tìm kiếm
        if beta <= alpha:
            break
    return v

def min_value(state, user_char, alpha, beta):
    if terminal(state):
        return utility(state, user_char)

    v = math.inf
    # Người chơi đóng vai trò Minimizer (Làm giảm điểm số của AI)
    for action in actions(state):
        v = min(v, max_value(result(state, action, user_char), user_char, alpha, beta))
        beta = min(beta, v)
        # Cắt tỉa: Nếu giá trị nhỏ hơn Alpha của lớp trên, dừng tìm kiếm
        if beta <= alpha:
            break
    return v

# --- GIAO DIỆN VÀ ĐIỀU KHIỂN ---

def print_board(board):
    """Hiển thị bàn cờ đẹp mắt với chỉ số hàng/cột"""
    print("\n      0   1   2")
    print("    -------------")
    for i, row in enumerate(board):
        row_str = f"  {i} |"
        for cell in row:
            char = cell if cell else "."
            row_str += f" {char} |"
        print(row_str)
        print("    -------------")

if __name__ == "__main__":
    board = initial_state()
    print("--- TRÒ CHƠI TIC-TAC-TOE (ALPHA-BETA PRUNING) ---")
    user_p = ""
    while user_p not in [X, O]:
        user_p = input("Bạn muốn cầm quân nào? (X/O): ").upper()

    turn = 0
    while not terminal(board):
        print_board(board)
        curr_p = player(board, user_p, turn)

        if curr_p == user_p:
            print(f"\n>> Lượt của bạn ({user_p})")
            try:
                r = int(input("Nhập hàng (0-2): "))
                c = int(input("Nhập cột (0-2): "))
                if (r, c) not in actions(board):
                    print("!!! Vị trí không hợp lệ hoặc đã có người đánh.")
                    continue
                board = result(board, (r, c), user_p)
            except ValueError:
                print("!!! Vui lòng chỉ nhập số nguyên từ 0 đến 2.")
                continue
        else:
            print(f"\n>> AI ({curr_p}) đang tính toán nước đi tối ưu...")
            move = alpha_beta_search(board, user_p)
            if move:
                board = result(board, move, curr_p)

        turn += 1

    # Hiển thị kết quả cuối cùng
    print_board(board)
    res = winner(board)
    if res:
        print(f"\nKẾT THÚC: {res} CHIẾN THẮNG!")
    else:
        print("\nKẾT THÚC: HÒA CỜ!")

--- TRÒ CHƠI TIC-TAC-TOE (ALPHA-BETA PRUNING) ---
Bạn muốn cầm quân nào? (X/O): O

      0   1   2
    -------------
  0 | . | . | . |
    -------------
  1 | . | . | . |
    -------------
  2 | . | . | . |
    -------------

>> Lượt của bạn (O)
Nhập hàng (0-2): 1
Nhập cột (0-2): 1

      0   1   2
    -------------
  0 | . | . | . |
    -------------
  1 | . | O | . |
    -------------
  2 | . | . | . |
    -------------

>> AI (X) đang tính toán nước đi tối ưu...

      0   1   2
    -------------
  0 | X | . | . |
    -------------
  1 | . | O | . |
    -------------
  2 | . | . | . |
    -------------

>> Lượt của bạn (O)
Nhập hàng (0-2): 2
Nhập cột (0-2): 2

      0   1   2
    -------------
  0 | X | . | . |
    -------------
  1 | . | O | . |
    -------------
  2 | . | . | O |
    -------------

>> AI (X) đang tính toán nước đi tối ưu...

      0   1   2
    -------------
  0 | X | . | . |
    -------------
  1 | . | O | . |
    -------------
  2 | X | . | O |
    -------------